# Querying the FlyRank Warehouse (~79M Rows) via DuckDB

**Dataset:** [`hf://datasets/FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)  
**Build ID:** `flyrank_pseudonymized_warehouse_release_v20260703`  
**Method:** Remote Parquet HTTP range queries with DuckDB (zero local raw file download)  

---

## 1. Setup & Authentication

> **Note:** In Google Colab, add your Hugging Face read token to **Secrets (🔑)** as `HF_TOKEN`.

In [ ]:
# Install dependencies
!pip install -q duckdb pandas pyarrow

import os
import duckdb

# In Colab, retrieve token from secrets; locally, use env var
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN', '')

con = duckdb.connect()
if hf_token:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}');")
con.execute("INSTALL httpfs; LOAD httpfs;")
print("DuckDB HTTPFS initialized successfully!")

## 2. Fast Metadata & Grain Probe

Queries against Parquet metadata are near-instant and do not transfer raw data rows.

In [ ]:
# Probe dim_clients and dim_content
con.sql("""
    SELECT 'dim_clients' AS table_name, COUNT(*) AS n_rows FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')
    UNION ALL
    SELECT 'dim_content' AS table_name, COUNT(*) AS n_rows FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
""").show()

## 3. Querying Partitions Safely (Mid-Panel Month)

> **Rule:** Develop queries on a middle partition (e.g., `month=2026-03`). Avoid scanning the full 78.8M rows repeatedly to avoid HTTP 429 rate limits.

In [ ]:
# Aggregate one month of daily telemetry into content-level features
month_features = con.sql("""
    SELECT 
        client_id,
        content_id,
        COUNT(DISTINCT report_date) AS active_days,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        AVG(gsc_avg_position) AS march_avg_position,
        SUM(ga4_sessions) AS march_sessions,
        ANY_VALUE(content_type) AS content_type
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE ga4_data_available = TRUE
    GROUP BY client_id, content_id
    HAVING march_impressions >= 100
""").df()

print(f"Extracted {len(month_features):,} content features from March 2026.")
month_features.head()

## 4. Full Warehouse Feature Construction & Local Caching

Run this scan once when your query logic is verified, and cache the aggregated result locally.

In [ ]:
# Construct 90-day feature window across all clients (Jan 1 -> Mar 31, 2026)
full_query = """
    SELECT 
        client_id,
        content_id,
        SUM(gsc_impressions) AS impressions_90d,
        SUM(gsc_clicks) AS clicks_90d,
        AVG(gsc_avg_position) AS avg_position,
        SUM(ga4_sessions) AS sessions_90d,
        SUM(ga4_engaged_sessions) AS engaged_sessions_90d,
        ANY_VALUE(content_type) AS content_type,
        ANY_VALUE(main_intent) AS main_intent
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet')
    WHERE report_date BETWEEN DATE '2026-01-01' AND DATE '2026-03-31'
      AND ga4_data_available = TRUE
    GROUP BY client_id, content_id
"""

print("Running full 90-day aggregation...")
df_cached = con.sql(full_query).df()

# Cache locally so future notebook reruns take seconds
os.makedirs('work/outputs', exist_ok=True)
df_cached.to_parquet('work/outputs/warehouse_features_cache.parquet', index=False)
print(f"Cached {len(df_cached):,} rows to work/outputs/warehouse_features_cache.parquet")